# Feast Feature Store on prokube

This notebook walks through the full Feast workflow:

1. **Setup** — install dependencies, configure the Feast client
2. **Generate data** — create sample driver statistics
3. **Define features** — entities, feature views, and on-demand transformations
4. **Register** — push definitions to the Feast registry
5. **Train** — retrieve historical features with point-in-time correctness
6. **Materialize** — push latest values to Redis for online serving
7. **Serve** — retrieve features at inference time

Everything happens inline in this notebook — no external Python files or CLI
commands needed.

### Prerequisites

Before running this notebook, make sure you have deployed:
- A **Redis** instance in your namespace (`redis-cr.yaml`)
- A **FeatureStore** CR in your namespace (`feast-cr.yaml`)
- The **feast-redis-config** secret (see `README.md` for setup steps)


---
## 1. Setup


In [ ]:
!pip install -q 'feast[redis]' scikit-learn

### Configure the Feast client

We build `feature_store.yaml` dynamically by reading the Redis connection
string from the `feast-redis-config` secret.

**About the registry path (`/tmp/registry.db`):**
The registry is a small SQLite database that stores feature definitions
(entities, feature views, data sources). We write it to `/tmp` which means
it does **not** survive a pod restart. That's fine — the registry only holds
*definitions*, not data. Your actual feature *values* live in Redis (online
store, persistent) and parquet files (offline store, on PVC). Just re-run
the "Define & Register" cell after a restart to recreate it.


In [ ]:
import base64
import subprocess
import yaml


def get_namespace():
    """Read the current namespace from the pod's service account."""
    try:
        with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
            return f.read().strip()
    except FileNotFoundError:
        return subprocess.check_output(
            ["kubectl", "config", "view", "--minify", "-o", "jsonpath={..namespace}"]
        ).decode().strip()


def get_redis_connection_string():
    """Read the Redis connection string from the feast-redis-config secret."""
    result = subprocess.run(
        ["kubectl", "get", "secret", "feast-redis-config",
         "-o", "jsonpath={.data.redis}"],
        capture_output=True, text=True, check=True,
    )
    raw = base64.b64decode(result.stdout).decode()
    return yaml.safe_load(raw)["connection_string"]


NAMESPACE = get_namespace()
REDIS_CONNECTION_STRING = get_redis_connection_string()
FEAST_PROJECT = "my_features"

feature_store_yaml = (
    f"project: {FEAST_PROJECT}\n"
    "provider: local\n"
    "offline_store:\n"
    "    type: file\n"
    "online_store:\n"
    "    type: redis\n"
    f"    connection_string: \"{REDIS_CONNECTION_STRING}\"\n"
    "registry:\n"
    "    registry_type: file\n"
    "    path: /tmp/registry.db\n"
    "auth:\n"
    "    type: no_auth\n"
    "entity_key_serialization_version: 3\n"
)

with open("feature_store.yaml", "w") as f:
    f.write(feature_store_yaml)

print("feature_store.yaml written")
print(f"Namespace: {NAMESPACE}")
print(f"Redis: {REDIS_CONNECTION_STRING.split(',')[0]}")


---
## 2. Generate sample data

We create a parquet file simulating hourly driver statistics over the past
7 days. In a real project this would come from your data pipeline.


In [ ]:
import datetime
import os

import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000
now = datetime.datetime.now()
timestamps = [now - datetime.timedelta(hours=i) for i in range(n)]

driver_df = pd.DataFrame({
    "driver_id": np.random.choice([1001, 1002, 1003, 1004, 1005], n),
    "event_timestamp": timestamps,
    "conv_rate": np.random.uniform(0.1, 1.0, n).astype(np.float32),
    "acc_rate": np.random.uniform(0.5, 1.0, n).astype(np.float32),
    "avg_daily_trips": np.random.randint(1, 50, n).astype(np.int64),
    "created": timestamps,
})

os.makedirs("data", exist_ok=True)
driver_df.to_parquet("data/driver_stats.parquet")
print(f"Created {n} rows for {driver_df['driver_id'].nunique()} drivers")
driver_df.head()


---
## 3. Define features

In Feast you define features as Python objects. There are two kinds:

- **FeatureView**: maps to columns in an existing data source (parquet file,
  database table, etc.). Feast stores and serves them — but doesn't compute
  anything. Your pipeline is responsible for producing the data.

- **On Demand Feature View (ODFV)**: a lightweight transformation that Feast
  executes at query time. It can combine existing features, add request-time
  inputs, or compute derived values. The transformation runs inline — during
  `get_historical_features()` and `get_online_features()` — so it's always
  consistent between training and serving.

Use FeatureViews for raw/precomputed data. Use ODFVs for derived features
that should be computed the same way everywhere.


In [ ]:
from datetime import timedelta

from feast import Entity, FeatureStore, FeatureView, Field, FileSource, RequestSource, ValueType
from feast.on_demand_feature_view import on_demand_feature_view
from feast.types import Float32, Float64, Int64

# ---------------------------------------------------------------------------
# Entity: the "primary key" for feature lookups.
# When you request features, you provide entity values (e.g. driver_id=1001).
# ---------------------------------------------------------------------------
driver = Entity(
    name="driver_id",
    value_type=ValueType.INT64,
    description="Unique driver identifier",
)

# ---------------------------------------------------------------------------
# Data source: points to where historical data lives.
# ---------------------------------------------------------------------------
driver_stats_source = FileSource(
    path="data/driver_stats.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

# ---------------------------------------------------------------------------
# FeatureView: declares which columns from the source are features.
# These are raw/precomputed values — Feast just stores and serves them.
# ---------------------------------------------------------------------------
driver_hourly_stats = FeatureView(
    name="driver_hourly_stats",
    entities=[driver],
    ttl=timedelta(days=7),
    schema=[
        Field(name="conv_rate", dtype=Float32),
        Field(name="acc_rate", dtype=Float32),
        Field(name="avg_daily_trips", dtype=Int64),
    ],
    source=driver_stats_source,
    online=True,
)

# ---------------------------------------------------------------------------
# On Demand Feature View: a derived feature computed by Feast at query time.
#
# "efficiency" = conv_rate / acc_rate
#
# This transformation runs automatically when you request this feature —
# both during training (get_historical_features) and serving
# (get_online_features). You define it once; Feast guarantees consistency.
# ---------------------------------------------------------------------------
@on_demand_feature_view(
    sources=[driver_hourly_stats],
    schema=[
        Field(name="efficiency", dtype=Float64),
    ],
    mode="pandas",
)
def driver_efficiency(features_df: pd.DataFrame) -> pd.DataFrame:
    df = pd.DataFrame()
    df["efficiency"] = features_df["conv_rate"] / features_df["acc_rate"]
    return df


print("Feature definitions created (not yet registered).")


---
## 4. Register features

`store.apply()` writes all definitions to the registry. After this, Feast
knows which features exist, where the data comes from, and how derived
features are computed.


In [ ]:
store = FeatureStore(repo_path=".")

store.apply([
    driver,
    driver_stats_source,
    driver_hourly_stats,
    driver_efficiency,
])

print("Registered:")
for fv in store.list_feature_views():
    print(f"  FeatureView: {fv.name}")
for odfv in store.list_on_demand_feature_views():
    print(f"  OnDemandFeatureView: {odfv.name}")


---
## 5. Retrieve historical features for training

`get_historical_features()` performs a **point-in-time join**: for each entity
row, it finds the most recent feature values *as of that timestamp*. This
prevents data leakage — you only see features that were available when the
event occurred.

Note how we request both raw features (`driver_hourly_stats:conv_rate`) and
the derived feature (`driver_efficiency:efficiency`). The ODFV runs
automatically — no extra code needed.

> **Note:** Feast will show a `RuntimeWarning` that on-demand feature views
> are experimental and don't scale well for offline retrieval. For this
> notebook-sized dataset that's irrelevant. For large-scale training data
> (millions of rows), precompute heavy features in your pipeline and use a
> regular `FeatureView` instead.


In [ ]:
# Build a query: "give me features for these drivers, as of this point in time."
# Each row says: I want to know the feature values for driver X at time T.
# Feast will find the most recent feature values that were available at that
# timestamp — this is the "point-in-time join" that prevents data leakage.

drivers_to_query = [1001, 1002, 1003, 1004, 1005]
query_timestamp = now  # "as of right now"

entity_df = pd.DataFrame({
    "driver_id": drivers_to_query,
    "event_timestamp": [query_timestamp] * len(drivers_to_query),
})

print("Query: get features for these drivers as of this timestamp:")
print(entity_df.to_string(index=False))

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
        "driver_efficiency:efficiency",          # <-- computed by Feast
    ],
).to_df()

print("Training data (point-in-time correct, incl. derived features):")
training_df


In [ ]:
from sklearn.linear_model import LinearRegression

FEATURE_COLS = ["acc_rate", "avg_daily_trips", "efficiency"]
TARGET = "conv_rate"

X = training_df[FEATURE_COLS].fillna(0)
y = training_df[TARGET].fillna(0)

model = LinearRegression().fit(X, y)
print(f"Model trained on {len(X)} samples.")


---
## 6. Materialize features to Redis

Materialization copies the latest feature values from the offline store
(parquet) into Redis for low-latency online serving.

In production you would run this on a schedule (e.g. hourly cron job).
Note: only `FeatureView` data is materialized. ODFV transformations are
computed on-the-fly at query time.


In [ ]:
from datetime import datetime, timedelta

store.materialize(
    start_date=datetime.now() - timedelta(days=7),
    end_date=datetime.now(),
)
print("Materialized to Redis.")


---
## 7. Online feature serving

Retrieve the latest feature values for specific drivers. This is what you
call at inference time. The raw features come from Redis; the `efficiency`
feature is computed on-the-fly by the ODFV — same formula as during training.


In [ ]:
online_features = store.get_online_features(
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
        "driver_efficiency:efficiency",
    ],
    entity_rows=[{"driver_id": 1001}, {"driver_id": 1002}],
).to_dict()

print("Online features (from Redis + ODFV):")
for k, v in online_features.items():
    print(f"  {k}: {v}")


In [ ]:
# Use online features for inference
inference_df = pd.DataFrame(online_features)
predictions = model.predict(inference_df[FEATURE_COLS])

for driver_id, pred in zip(inference_df["driver_id"], predictions):
    print(f"Driver {driver_id}: predicted conv_rate = {pred:.4f}")


---
## Summary

| Step | API | What happens |
|------|-----|-------------|
| Define | Python objects (Entity, FeatureView, ODFV) | Declare what features exist and how derived ones are computed |
| Register | `store.apply([...])` | Write definitions to the registry (once per session) |
| Train | `store.get_historical_features()` | Point-in-time join from parquet; ODFVs run inline |
| Materialize | `store.materialize()` | Push latest raw values to Redis |
| Serve | `store.get_online_features()` | Sub-ms lookup from Redis; ODFVs run inline |

### FeatureView vs On Demand Feature View

| | FeatureView | On Demand Feature View |
|-|-------------|------------------------|
| Data | Precomputed in your pipeline | Computed by Feast at query time |
| Materialized to Redis? | Yes | No (computed on-the-fly) |
| Good for | Raw/heavy features | Lightweight derived features, request-time inputs |
| Consistency | You ensure pipeline runs | Feast guarantees same logic in training & serving |

### About the registry

The registry (`/tmp/registry.db`) stores only *definitions* — not feature
values. It is ephemeral in this notebook setup. If your pod restarts, re-run
the "Define & Register" cells. Your feature *data* in Redis and on PVC is
not affected.
